# AL_0048 photostim grid — TF fits & bidirectional drive

Interactive companion to the `bilateral/grid/` modules. Heavy lifting stays in the modules
(so batch runs stay reproducible); this notebook loads their caches and explores/plots inline.

## Where the code lives — `brain_paper/bilateral/grid/`

| file | role |
|---|---|
| `config.py` | session, paths, windows, registration constants (`BREGMA_PX`, `PX_PER_MM_*`, `AMP_SEL`) |
| `loader.py` | raw Timeline → stim onsets + galvo positions; block segmentation; power select |
| `calibration.py` | galvo volts → mm-from-bregma |
| `analysis.py` | per-site dF/F, rise-tau, full-frame spatial snapshots |
| `cross_response.py` | **the 52×52 cross-response tensor** `H[s,r,t]` + per-trial cache (one SVD pass) |
| `tf_fit.py` | **TF fitting**: per-amp (`fit_all`, `fit_all_amps`) and shared amp-as-input (`fit_shared`) |
| `tf_sweep.py` | model-class + hyperparameter sweep, scored on held-out CV-R² |
| `tf_matrix.py` | gain matrix, amp-comparison, exemplars, spatial response fields |
| `bidirectional.py` | FDR-tested sites drivable both up and down |
| `interactive_grid.py` | clickable brain: selector map → efferent field with TF overlays |
| `network_id.py` | dynamical network model (graph-Laplacian / wave / delay-DMD) |
| `run_grid.py`, `run_grid_all.py` | batch entry points |
| `plots.py` | site/timecourse/tau/spatial/raster figures |
| `legacy/` | pre-refactor monolith + MATLAB port + source notebook |

Caches live in `brain_paper/data/` (all gitignored, all regenerable):
`grid_cross_response*.npz`, `grid_trials*.npz`, `grid_tf_fits.npz` (single amp),
`grid_tf_fits_2amp.npz` (independent per amp), `grid_tf_fits_shared.npz` (shared amp-as-input),
`grid_bidirectional.npz`, `grid_brain.npz`.

**Regenerate** (from `bilateral/grid/`, repo `.venv`):
```
python cross_response.py          # tensor + trials cache (slow: one SVD pass)
python tf_fit.py all              # single-amp fits
python tf_fit.py 2amp             # independent fit per amplitude
python tf_fit.py shared           # ONE TF, amplitude as input
python tf_matrix.py               # figures
python bidirectional.py           # bidirectional sites
python interactive_grid.py        # clickable viewer (add 'save' for PNGs)
```

In [ ]:
%matplotlib inline
import sys, os
from pathlib import Path

GRID = Path.cwd() if Path.cwd().name == "grid" else Path("bilateral/grid").resolve()
os.chdir(GRID); sys.path.insert(0, str(GRID))
DATA = GRID.parents[1] / "data"

import numpy as np
import matplotlib.pyplot as plt

# NOTE: import only backend-safe modules here. `tf_matrix`, `plots` and `bidirectional`
# call matplotlib.use("Agg") at import time, which disables inline figures.
import tf_fit, cross_response

print("grid dir:", GRID)
print("caches:", *[p.name for p in sorted(DATA.glob("grid_*.npz"))], sep="\n  ")

## 1. Load the fits

Three fit flavours, all on the same 52×52 pairs:

- **`grid_tf_fits.npz`** — single amplitude (`AMP_SEL`, currently 2.0).
- **`grid_tf_fits_2amp.npz`** — an *independent* TF per amplitude. This is the **LTI test**: are
  the poles amplitude-invariant? (They are: median τ ratio ≈ 0.95.)
- **`grid_tf_fits_shared.npz`** — **one** TF for both amplitudes, with the laser amplitude as the
  **input**: response to amp `a` is `a·h(t)`. No free per-amp gain — the bigger drive is what
  makes the response bigger. This *forces* a 2× ratio between amp 1.0 and 2.0 while the measured
  focal ratio is ~1.4–1.5×, so its residual **measures saturation** rather than indicating a bad fit.

In [ ]:
def load(name):
    z = np.load(DATA / name, allow_pickle=True)
    return {k: z[k] for k in z.files}

def load_opt(name, how):
    """Tolerant load: returns None (with a hint) if the cache isn't generated yet,
    so the notebook still runs end-to-end and dependent cells skip cleanly."""
    if not (DATA / name).exists():
        print(f"[missing] {name}")
        print(f"          generate with:  {how}")
        print(f"          cells needing it will be skipped.")
        return None
    return load(name)

Z1 = load("grid_tf_fits.npz")            # single amp
Z2 = load("grid_tf_fits_2amp.npz")       # independent TF per amp
ZS = load_opt("grid_tf_fits_shared.npz", "python tf_fit.py shared")   # shared, amp-as-input
have_shared = ZS is not None

sites, window = Z2["sites"], Z2["window"]
amps = Z2["amps"]
nS = len(sites)
print()
print(f"{nS} sites, amps={list(amps)}, window {window[0]:.2f}..{window[-1]:.2f}s, "
      f"model={Z1.get('model')}, shared_fit={'yes' if have_shared else 'NOT YET'}")

def idx(mx, my):
    """Site index nearest a (ML, AP) coordinate in mm from bregma."""
    return int(np.argmin(np.hypot(sites[:, 0] - mx, sites[:, 1] - my)))

def lab(i):
    return f"({sites[i,0]:+.1f},{sites[i,1]:+.0f})"


## 2. ⚠ Reliability first — in-sample R² is NOT a quality metric here

The TF model class is **damped sinusoids** (`A·e^{−t/τ}·sin(ωt+φ)`, complex pole pairs). It is
flexible enough to fit *noise* at R² ≈ 0.81. Always gate on **held-out CV-R²** (split-half over
trials), never on R². The cell below shows why.

In [ ]:
H, r2, cv = Z1["H"], Z1["r2"], Z1["cvr2"]
post = window >= 0
strength = np.abs(H[:, :, post]).max(2).ravel()
o = np.argsort(-strength)

print(f"{'subset':<22}{'n':>6}{'medR2':>8}{'medCV':>9}{'CV>0':>8}")
for name, sel in [("top 150 (real signal)", o[:150]), ("top 500", o[:500]), ("rest (weak/noise)", o[500:])]:
    print(f"{name:<22}{len(sel):>6}{np.nanmedian(r2.ravel()[sel]):>8.3f}"
          f"{np.nanmedian(cv.ravel()[sel]):>9.3f}{100*np.nanmean(cv.ravel()[sel]>0):>7.1f}%")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), constrained_layout=True)
ax[0].hist(r2.ravel(), bins=60, color="steelblue"); ax[0].set(title="in-sample R² (piles at 1.0 = overfit)", xlabel="R²")
ax[1].hist(np.clip(cv.ravel(), -5, 1), bins=60, color="indianred"); ax[1].axvline(0, c="k", ls="--")
ax[1].set(title="held-out CV-R² (the honest metric)", xlabel="CV-R² (clipped at -5)")
plt.show()

GOOD = cv > 0     # reuse this mask everywhere below
print(f"\npairs that generalize (CV-R²>0): {GOOD.sum()} / {GOOD.size}")

## 3. Inspect any pair: data + fit, both amplitudes

Change `S`/`R` to any (ML, AP) pair. Solid = trial mean, dashed = fit.
The **shared** fit (one `h`, amp as input) is overlaid against the **independent** per-amp fits.

In [ ]:
S, R = idx(-3.5, -1), idx(-1.5, +2)      # <-- edit: stim site, readout site

ncol = 2 if have_shared else 1
fig, ax = plt.subplots(1, ncol, figsize=(6*ncol, 4), squeeze=False, constrained_layout=True)
ax = ax[0]
cols = ["#1f77b4", "#d62728"]
for ai, c_ in enumerate(cols[:len(amps)]):
    ax[0].plot(window, Z2["H"][ai, S, R], c=c_, lw=1.4, label=f"amp {amps[ai]:.1f} data")
    ax[0].plot(window, Z2["yhat"][ai, S, R], c=c_, lw=1.2, ls="--")
    if have_shared:
        ax[1].plot(window, ZS["H"][ai, S, R], c=c_, lw=1.4, label=f"amp {amps[ai]:.1f} data")
        ax[1].plot(window, ZS["yhat"][ai, S, R], c=c_, lw=1.2, ls="--")
titles = ["INDEPENDENT fit per amp", "SHARED fit (amp = input, forces ×2)"]
for a, t in zip(ax, titles[:ncol]):
    a.axvline(0, c="grey", lw=0.6); a.axhline(0, c="k", lw=0.4, ls=":")
    a.set(xlim=(-0.2, 0.6), xlabel="t (s)", ylabel="dF/F", title=t); a.legend(fontsize=8)
fig.suptitle(f"stim {lab(S)} → readout {lab(R)}")
plt.show()

print(f"independent : order {Z2['order'][:,S,R]}  R2 {np.round(Z2['r2'][:,S,R],3)}  "
      f"CV {np.round(Z2['cvr2'][:,S,R],3)}")
if have_shared:
    print(f"shared      : order {ZS['order'][S,R]}  pooled R2 {ZS['r2'][S,R]:.3f}  "
          f"per-amp R2 {np.round(ZS['r2_amp'][:,S,R],3)}  CV {ZS['cvr2'][S,R]:.3f}")
    om = ZS["om"][S, R]; om = om[np.isfinite(om)]
    tv = ZS["tau"][S, R]; tv = tv[np.isfinite(tv)]
    print(f"shared modes: tau(ms) {np.round(tv*1e3,1)}  freq(Hz) {np.round(om/(2*np.pi),2)}")


## 4. Does one shared TF explain both amplitudes?

This is the amplitude-linearity question, asked properly: the shared model has the *same* number
of dynamic parameters as a single-amp fit and must explain **both** amplitudes with the input
doing the scaling. Compare its per-amp R² to the independent per-amp fits — the gap is the price
of enforcing strict amplitude-linearity, i.e. the **saturation**.

In [ ]:
if not have_shared:
    print("skipped - run `python tf_fit.py shared` first (writes grid_tf_fits_shared.npz).")
else:
    m = (ZS["cvr2"] > 0) & np.all(np.isfinite(Z2["r2"]), 0)
    print(f"pairs compared (shared CV-R2>0): {m.sum()}")
    print()
    print(f"{'':<14}{'shared R2':>11}{'indep R2':>11}{'gap':>8}")
    for ai in range(len(amps)):
        s_, i_ = ZS["r2_amp"][ai][m], Z2["r2"][ai][m]
        print(f"amp {amps[ai]:<10.1f}{np.nanmedian(s_):>11.3f}{np.nanmedian(i_):>11.3f}"
              f"{np.nanmedian(i_-s_):>8.3f}")
    fig, ax = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    for ai in range(len(amps)):
        ax[0].scatter(Z2["r2"][ai][m], ZS["r2_amp"][ai][m], s=12, alpha=0.5,
                      label=f"amp {amps[ai]:.1f}")
    ax[0].plot([0, 1], [0, 1], "k--", lw=1)
    ax[0].set(xlabel="independent per-amp R²", ylabel="shared (amp-as-input) R²",
              title="cost of forcing one TF + linear input"); ax[0].legend(fontsize=8)
    di = np.arange(nS)
    dm = (Z2["cvr2"][0][di, di] > 0.2) & (Z2["cvr2"][1][di, di] > 0.2)
    ratio = np.abs(Z2["gain"][1][di, di][dm]) / np.abs(Z2["gain"][0][di, di][dm])
    ax[1].hist(ratio, bins=12, color="seagreen")
    ax[1].axvline(np.median(ratio), c="k", lw=1.5, label=f"median {np.median(ratio):.2f}")
    ax[1].axvline(amps[1]/amps[0], c="r", ls="--", lw=1.5, label=f"linear {amps[1]/amps[0]:.1f}")
    ax[1].set(xlabel="focal gain ratio amp2/amp1", ylabel="# sites",
              title="measured dose-response (on-site, CV-gated)"); ax[1].legend(fontsize=8)
    plt.show()


## 5. Spatial view — responses at their cortical location

Efferent field: stimulate one site, colour every readout at its true (ML, AP) by signed peak gain.
Unreliable pairs (CV-R² ≤ 0) are drawn hollow so overfit noise doesn't read as coupling.

In [ ]:
STIM = idx(-3.5, -1)                      # <-- edit
ai = len(amps) - 1                         # highest amp
g = Z2["gain"][ai][STIM]
ok = Z2["cvr2"][ai][STIM] > 0
clim = np.nanpercentile(np.abs(Z2["gain"][ai]), 97)

fig, ax = plt.subplots(figsize=(5.5, 5.2), constrained_layout=True)
ax.scatter(sites[~ok, 0], sites[~ok, 1], facecolors="none", ec="0.7", s=150, lw=1.0)
sc = ax.scatter(sites[ok, 0], sites[ok, 1], c=g[ok], cmap="bwr", vmin=-clim, vmax=clim,
                s=170, ec="0.3", lw=0.5)
ax.scatter(*sites[STIM], s=380, facecolors="none", ec="k", lw=2.2)
ax.axvline(0, c="0.6", lw=0.7, ls="--"); ax.plot(0, 0, "+", c="lime", ms=10, mew=1.6)
ax.set(aspect="equal", xlabel="ML from bregma (mm)", ylabel="AP from bregma (mm)",
       title=f"efferent field — stim {lab(STIM)}, amp {amps[ai]:.1f}\n(hollow = CV-R²≤0, not trustworthy)")
fig.colorbar(sc, ax=ax, shrink=0.8, label="signed peak dF/F")
plt.show()

## 6. Bidirectionally drivable sites

From `bidirectional.py` — **empirical**, deliberately independent of the TF fits. Per-trial dF/F,
each trial baselined to its own pre-onset mean, tested in two windows (the response is biphasic:
excitatory peak ~70 ms, suppression trough ~140–240 ms), one-sample t-test per pair, **BH-FDR over
all pairs × windows before any max is taken**, then a ≥0.5% dF/F practical floor.

In [ ]:
B = load("grid_bidirectional.npz")
bd, ap, an, wp, wn = B["bidir"], B["arg_pos"], B["arg_neg"], B["win_pos"], B["win_neg"]
wnames = [str(w) for w in B["windows"]]
print(f"bidirectional: {bd.sum()}/{len(sites)}")
print(f"  same driver for +/- (would be ONE biphasic response, not independent): {(bd & (ap == an)).sum()}")
print(f"  different drivers (genuine independent actuators):                     {(bd & (ap != an)).sum()}")
print(f"  both directions within the SAME time window:                           {(bd & (wp == wn)).sum()}")
print("\n-> UP is driven in the early lobe, DOWN in the late lobe for almost every site:")
print("   the two actuation directions have DIFFERENT LATENCIES (~70 ms vs ~140-240 ms),")
print("   which a closed-loop controller must model as asymmetric actuation delay.")

fig, ax = plt.subplots(figsize=(5.5, 5.2), constrained_layout=True)
sc = ax.scatter(sites[bd, 0], sites[bd, 1], c=B["rng"][bd]*100, cmap="viridis", s=210, ec="k", lw=0.6)
ax.scatter(sites[~bd, 0], sites[~bd, 1], facecolors="none", ec="0.6", s=150, lw=1.0)
ax.axvline(0, c="0.6", lw=0.7, ls="--"); ax.plot(0, 0, "+", c="lime", ms=10, mew=1.6)
ax.set(aspect="equal", xlabel="ML from bregma (mm)", ylabel="AP from bregma (mm)",
       title=f"bidirectionally drivable: {bd.sum()}/{len(sites)}")
fig.colorbar(sc, ax=ax, shrink=0.8, label="dynamic range (% dF/F)")
plt.show()

## 7. Standing caveats

1. **Never gate on in-sample R².** Under the damped-sinusoid model, noise pairs reach R² ≈ 0.81
   with CV-R² ≈ −2.1. Only ~25% of the 2704 pairs generalize at all. Use `cvr2 > 0`.
2. **Quote the gain *ratio*, not the slope.** Across every model/gate variant the median focal
   ratio held at 1.37–1.49, while the through-origin slope swung 0.83–1.25. Sublinear either way.
3. **Excitatory drive is broad, inhibitory is focal.** The broad positive field is exactly where a
   single-wavelength hemodynamic confound would live, so the *down* direction is the more
   trustworthy of the two.
4. **Fitted mode frequencies are median ~2.7 Hz** (10–90%: 0.8–7.5 Hz). They overlap `network_id`'s
   7–8 Hz graph-wave band only in the top decile — that is tail overlap, *not* corroboration.
5. **Some τ rail at the 3.0 s bound**, where a slow mode acts as a DC/trend term over a 0.6 s fit
   window rather than real dynamics. Open item: tighten the bound or penalise near-DC modes.
6. **n = 1 session** (AL_0048 2026-07-10). Nothing here is replicated yet.